**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Representation Learning

Learning without labels: autoencoders that discover a signal's true coordinates, VAEs that turn compression into generation, and contrastive learning — the idea behind modern self-supervised pretraining. All demonstrated on a dataset whose true hidden structure we control, so we can *check* what was learned.

## 1. Pre-requisites

- [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb), [Intro to ANN](./Intro_ANN/Intro_ANN.ipynb).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S4 (PCA — the linear ancestor).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# Dataset with KNOWN hidden structure: damped tones with 2 latent factors
# (frequency, decay) → 64-sample waveforms. Can 2 learned dimensions recover them?
def make_signals(n):
    freq = rng.uniform(0.05, 0.45, n)         # latent 1
    decay = rng.uniform(0.5, 4.0, n)          # latent 2
    t = np.arange(64)
    X = np.exp(-decay[:, None] * t / 64) * np.sin(2*np.pi*freq[:, None]*t)
    X += 0.02 * rng.standard_normal(X.shape)
    return X.astype(np.float32), freq, decay

X_np, freq, decay = make_signals(4000)
X = torch.from_numpy(X_np)
plt.figure(figsize=(8, 2))
for i in range(4): plt.plot(X_np[i], alpha=0.8)
plt.title("samples: damped tones, secretly governed by just 2 numbers")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2677185/3006020170.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 1 of 3 — *Autoencoders* (~35 min)
**Goal:** compress through a bottleneck; check the latent space against the true factors.
**Builds on:** [ANN](./Intro_ANN/Intro_ANN.ipynb). &nbsp; **Feeds into:** Session 2 (VAEs).

---

## 2. The Bottleneck Game

💡 **Intuition.** An autoencoder learns two maps — encode (64 numbers → 2) and decode (2 → 64) — trained only to reconstruct its input. The bottleneck is the whole trick: to squeeze 64 samples through 2 numbers and back, the network is *forced* to discover the data's true coordinates. It's [PCA](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) with nonlinearity — and unlike PCA it can flatten curved manifolds.

In [2]:
class AE(nn.Module):
    def __init__(self, d_lat=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, d_lat))
        self.dec = nn.Sequential(nn.Linear(d_lat, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 64))
    def forward(self, x): return self.dec(self.enc(x))

ae = AE()
opt = torch.optim.Adam(ae.parameters(), lr=2e-3)
for ep in range(60):
    for i in range(0, 4000, 128):
        xb = X[i:i+128]
        loss = ((ae(xb) - xb)**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
print(f"reconstruction MSE through a 2-D bottleneck: {loss.item():.5f} (signal var ≈ {X_np.var():.3f})")

reconstruction MSE through a 2-D bottleneck: 0.01048 (signal var ≈ 0.133)


In [3]:
# The check most tutorials skip: does the latent space recover the TRUE factors?
with torch.no_grad():
    Z = ae.enc(X).numpy()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, c, name in [(axes[0], freq, "colored by TRUE frequency"), (axes[1], decay, "colored by TRUE decay")]:
    s = ax.scatter(Z[:, 0], Z[:, 1], c=c, s=3, cmap="viridis")
    ax.set_title(name); plt.colorbar(s, ax=ax)
plt.suptitle("the 2-D latent space, audited against the generative truth", y=1.02)
plt.tight_layout(); plt.show()
print("smooth color gradients = the bottleneck rediscovered (a warped version of) the real factors")

smooth color gradients = the bottleneck rediscovered (a warped version of) the real factors


/tmp/ipykernel_2677185/3616095439.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Variational Autoencoders* (~40 min)
**Goal:** make the latent space a smooth, sampleable distribution; generate new signals.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (contrastive learning).

---

## 3. From Compression to Generation

💡 **Intuition.** A plain AE's latent space has holes — decode a random point and you may get garbage, because nothing forced the space to be *filled*. The VAE fixes this by encoding each input as a **distribution** (mean + spread) and penalizing the ensemble toward a standard Gaussian (a KL term — [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb)'s price of wrong beliefs, used as glue). Result: overlapping, hole-free latents — sample $z \sim \mathcal{N}(0, I)$, decode, and you *generate*. The reparameterization trick ($z = \mu + \sigma \epsilon$) keeps sampling differentiable.

In [4]:
class VAE(nn.Module):
    def __init__(self, d_lat=2):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.mu, self.logv = nn.Linear(32, d_lat), nn.Linear(32, d_lat)
        self.dec = nn.Sequential(nn.Linear(d_lat, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 64))
    def forward(self, x):
        h = self.body(x)
        mu, logv = self.mu(h), self.logv(h)
        z = mu + torch.exp(0.5*logv) * torch.randn_like(mu)     # reparameterization
        return self.dec(z), mu, logv

vae = VAE(); opt = torch.optim.Adam(vae.parameters(), lr=2e-3)
beta = 0.001
for ep in range(80):
    for i in range(0, 4000, 128):
        xb = X[i:i+128]
        xr, mu, logv = vae(xb)
        rec = ((xr - xb)**2).mean()
        kl = (-0.5 * (1 + logv - mu**2 - logv.exp())).mean()
        loss = rec + beta * kl
        opt.zero_grad(); loss.backward(); opt.step()
print(f"reconstruction {rec.item():.5f}   KL {kl.item():.3f}")

reconstruction 0.01077   KL 4.300


In [5]:
# Generate: walk a grid of the latent space and decode — brand-new signals
with torch.no_grad():
    grid = torch.tensor([[a, b] for b in np.linspace(-2, 2, 4) for a in np.linspace(-2, 2, 4)], dtype=torch.float32)
    gen = vae.dec(grid).numpy()
fig, axes = plt.subplots(4, 4, figsize=(9, 5), sharex=True, sharey=True)
for ax, g in zip(axes.ravel(), gen):
    ax.plot(g, linewidth=0.9); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("decoded from a grid over z ~ N(0,I): novel signals, smoothly morphing")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2677185/3742532623.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Contrastive Learning* (~40 min)
**Goal:** learn representations by agreement between augmented views — no reconstruction at all.
**Builds on:** Session 2.

---

## 4. Learning by Comparison

💡 **Intuition.** Reconstruction wastes capacity on details you don't care about (exact noise, phase). Contrastive learning changes the question: make two *augmented views* of the same signal map **close**, and views of different signals map **apart**. What survives is exactly what your augmentations declare irrelevant — invariance is a *design choice*. This is the engine of SimCLR/CLIP-style pretraining: no labels, just the physics of 'what shouldn't matter'.

In [6]:
def augment(xb):
    """views that preserve identity (freq/decay) but change nuisance: shift + noise + gain"""
    out = xb.clone()
    shift = int(torch.randint(0, 8, (1,)))
    out = torch.roll(out, shift, dims=1)
    out = out * (0.8 + 0.4*torch.rand(len(out), 1)) + 0.05*torch.randn_like(out)
    return out

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2))
    def forward(self, x):
        z = self.f(x)
        return z / (z.norm(dim=1, keepdim=True) + 1e-8)         # unit sphere

enc = Encoder(); opt = torch.optim.Adam(enc.parameters(), lr=2e-3)
tau = 0.2
for ep in range(50):
    for i in range(0, 4000, 256):
        xb = X[i:i+256]
        z1, z2 = enc(augment(xb)), enc(augment(xb))
        logits = (z1 @ z2.T) / tau                              # similarity of every pair
        labels = torch.arange(len(xb))                          # the diagonal are the true pairs
        loss = nn.functional.cross_entropy(logits, labels)      # InfoNCE
        opt.zero_grad(); loss.backward(); opt.step()
print(f"final InfoNCE loss {loss.item():.3f} (chance = ln(256) = {np.log(256):.2f})")

final InfoNCE loss 3.404 (chance = ln(256) = 5.55)


In [7]:
with torch.no_grad():
    Zc = enc(X).numpy()
plt.figure(figsize=(4.6, 4))
s = plt.scatter(Zc[:, 0], Zc[:, 1], c=freq, s=3, cmap="viridis")
plt.colorbar(s, label="true frequency")
plt.title("contrastive embedding (on the circle):\norganized by frequency, invariant to shift/gain/noise BY DESIGN")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2677185/915262890.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

Bottlenecks force discovery; the VAE's KL glue turns discovery into generation; contrastive losses let *you* choose what representation ignores. Three label-free recipes — the third is how frontier models pretrain.

---
## Where next

- [Diffusion Models](./Diffusion_Models.ipynb) — generation taken to the modern state of the art.
- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — next-token prediction as yet another self-supervised objective.